# **Taller Despliegue Flask API**

## PARTE II: Despliegue remoto en Render

En la parte anterior desplegamos nuestra API Flask de forma local. Esto nos permitió probarla en nuestro propio equipo, pero para que otros puedan consumirla necesitamos alojarla en un servidor accesible desde internet.

En esta parte vamos a desplegar la misma API en **Render**, una plataforma cloud que ofrece un plan gratuito suficiente para este tipo de proyectos.

Al finalizar tendremos una URL pública con la que cualquier cliente (una web, una app móvil, un notebook...) podrá hacer peticiones a nuestro modelo.

### Requisitos previos

Antes de empezar necesitamos:

1. **Cuenta en GitHub** con el proyecto subido en un repositorio (público o privado).
2. **Cuenta en Render** — puedes registrarte gratis en [render.com](https://render.com) con tu cuenta de GitHub.
3. **El proyecto preparado** con los ficheros adicionales que veremos a continuación.

### Procedimiento

**#1. Preparar el proyecto para el despliegue**

Render necesita saber qué dependencias instalar y cómo arrancar nuestra aplicación. Para ello debemos añadir dos ficheros al proyecto:

**`requirements.txt`** — lista todas las librerías necesarias. Render las instalará automáticamente antes de arrancar el servicio.

```
flask
gunicorn
pandas
numpy
scikit-learn==1.5.2
```

> **Fíjate:** `gunicorn` aparece en el `requirements.txt` pero no en los imports de `app_model.py`. Esto es porque Gunicorn no es una librería que uses en tu código, sino un **servidor web** que se ejecuta desde la línea de comandos y es quien se encarga de arrancar tu aplicación Flask. En local usábamos `app.run()` (el servidor de desarrollo de Flask), pero en producción Gunicorn actúa como intermediario entre las peticiones HTTP entrantes y tu app, siendo más estable y eficiente. Por eso lo instalamos con `pip` pero no lo importamos.

Además, debemos asegurarnos de que `app_model.py` tiene la línea `os.chdir` **descomentada**, para que los ficheros (`ad_model.pkl`, `data/`) se resuelvan correctamente independientemente del directorio desde el que arranque el servidor:

```python
os.chdir(os.path.dirname(__file__))
```

**#2. Subir el proyecto a GitHub**

Si aún no tienes el proyecto en un repositorio, inicialízalo y súbelo:

```bash
git init
git add .
git commit -m "first commit"
git remote add origin https://github.com/tu-usuario/tu-repositorio.git
git push -u origin main
```

Asegúrate de que el repositorio incluye: `app_model.py`, `model.py`, `requirements.txt`, `ad_model.pkl` y la carpeta `data/`.

**#3. Crear el servicio web en Render**

1. Entra en [render.com](https://render.com) e inicia sesión.
2. Haz clic en **New → Web Service**.
3. Conecta tu cuenta de GitHub y selecciona el repositorio del proyecto.
4. Rellena la configuración del servicio:

| Campo | Valor |
|---|---|
| **Name** | `api-advertising` (o el que prefieras) |
| **Region** | La más cercana a tu ubicación |
| **Branch** | `main` |
| **Build Command** | `pip install -r requirements.txt` |
| **Start Command** | `gunicorn app_model:app` |
| **Instance Type** | Free |

5. Haz clic en **Create Web Service**.

Render clonará el repositorio, instalará las dependencias y arrancará el servidor. El proceso tarda unos minutos. Una vez completado verás el estado **Live** y una URL del tipo `https://api-advertising.onrender.com`.

**#4. Probar la API remota**

Con la URL que nos proporciona Render ya podemos consumir la API desde cualquier sitio. Sustituye `TU_URL` por la URL real de tu servicio.

In [13]:
import requests

BASE_URL = "https://taller-despliegue-api-1.onrender.com"

In [14]:
# Endpoint home
response = requests.get(BASE_URL + '/')
print(response.status_code)
print(response.text)

200
Bienvenido a mi API del modelo advertising


In [16]:
params = {'tv': 150, 'radio': 30, 'newspaper': 20}

response = requests.post(BASE_URL + '/api/v1/predict', params=params)

print("status:", response.status_code)
print("texto:", response.text)

status: 405
texto: <!doctype html>
<html lang=en>
<title>405 Method Not Allowed</title>
<h1>Method Not Allowed</h1>
<p>The method is not allowed for the requested URL.</p>



In [15]:
# Endpoint predict
params = {'tv': 150, 'radio': 30, 'newspaper': 20}
response = requests.post(BASE_URL + '/api/v1/predict', params=params)
print(response.status_code)
print(response.json())

405


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# Endpoint predict con un valor faltante (será imputado por el pipeline)
params = {'tv': 150, 'radio': 30}
response = requests.post(BASE_URL + '/api/v1/predict', params=params)
print(response.status_code)
print(response.json())

In [ ]:
# Endpoint retrain
response = requests.get(BASE_URL + '/api/v1/retrain')
print(response.status_code)
print(response.text)

**Nota sobre el plan gratuito de Render:** los servicios gratuitos se \"duermen\" tras 15 minutos de inactividad. La primera petición después de un periodo sin uso puede tardar entre 30 y 60 segundos en responder mientras el servicio arranca de nuevo. Es un comportamiento normal en el plan free.

### **¡Enhorabuena!** Has desplegado tu API de manera remota en Render :)